# 02 · ALL vs STRICT: how much does the picture change? (RQ2)

Person 2's log (8 November 2026) records an executed `notebooks/02_all_vs_strict.ipynb`, but the file was not in
the shared archive. This notebook **re-creates** the analysis from the outputs of `src/enrich.py`, so that every
RQ2 number in the report can be re-run. If Person 2 still has the original notebook, add it back and keep this
one as a cross-check.

Inputs (read only): `data/processed/works_layers.csv`, `data/processed/authorships_layers.csv`.

**Definitions.**
- ALL = every clean paper. STRICT = `in_strict` (a Scopus venue covered in the year of publication).
- Rankings count papers (full counting): an institution or country counts once per paper.
- Rank stability = Spearman correlation of paper counts over the union of the two top-20 lists. It is computed
  as the Pearson correlation of average ranks, so no scipy is needed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P = ROOT / "data" / "processed"
w = pd.read_csv(P / "works_layers.csv", dtype={"scopus_id": str})
a = pd.read_csv(P / "authorships_layers.csv")
pi = a.dropna(subset=["institution_id"]).drop_duplicates(["work_id", "institution_id"])
pc = a.dropna(subset=["institution_country"]).drop_duplicates(["work_id", "institution_country"])
VARIANTS = ["in_strict", "in_strict_issn_only", "in_strict_active", "in_strict_lag", "in_strict_q12"]
print(len(w), "papers")

10208 papers


## 1. Size of the layers

In [2]:
sizes = pd.DataFrame({"papers": [len(w), int(w["in_scopus"].sum())] + [int(w[v].sum()) for v in VARIANTS]},
                     index=["ALL", "in Scopus (any year)"] + VARIANTS)
sizes["share_of_ALL"] = (sizes["papers"] / len(w)).round(3)
sizes

,papers,share_of_ALL
ALL,10208,1.000
in Scopus (any year),6089,0.596
in_strict,5927,0.581
in_strict_issn_only,5262,0.515
in_strict_active,5824,0.571
in_strict_lag,5987,0.587
in_strict_q12,3886,0.381


In [3]:
by_type = w.groupby("type")["in_strict"].agg(papers="size", strict="sum")
by_type["survival"] = (by_type["strict"] / by_type["papers"]).round(2)
by_year = w.groupby("year")["in_strict"].agg(papers="size", strict="sum")
by_year["survival"] = (by_year["strict"] / by_year["papers"]).round(2)
display(by_type.sort_values("survival", ascending=False))
by_year

,papers,strict,survival
type,,,
review,340,291,0.86
article,5777,3954,0.68
conference-paper,2963,1539,0.52
book-chapter,629,143,0.23
preprint,499,0,0.00


,papers,strict,survival
year,,,
2020,471,299,0.63
2021,562,360,0.64
2022,637,421,0.66
2023,1316,787,0.60
2024,2730,1661,0.61
2025,4492,2399,0.53


## 2. Rankings in ALL and STRICT

In [4]:
def spearman(x, y):
    return float(pd.Series(x).rank().corr(pd.Series(y).rank()))

def compare(df, key, strict_col="in_strict", top=20, label=None):
    A = df.groupby(key)["work_id"].nunique()
    S = df[df[strict_col]].groupby(key)["work_id"].nunique().reindex(A.index, fill_value=0)
    ra = A.rank(ascending=False, method="min")
    rs = S.rank(ascending=False, method="min")
    topA = A.sort_values(ascending=False).index[:top]
    topS = S.sort_values(ascending=False).index[:top]
    union = topA.union(topS)
    table = pd.DataFrame({"papers_ALL": A[topA], "rank_ALL": ra[topA].astype(int),
                          "papers_STRICT": S[topA], "rank_STRICT": rs[topA].astype(int)})
    if label is not None:
        table.insert(0, "name", label.reindex(topA).values)
    stats = {"shared_top": len(set(topA) & set(topS)),
             "jaccard": round(len(set(topA) & set(topS)) / len(union), 3),
             "spearman_union_top": round(spearman(A[union], S[union]), 2)}
    return table, stats, topS

eu_c, eu_i = pc[pc["is_eu"]], pi[pi["is_eu"]]
countries, s_c, _ = compare(eu_c, "institution_country")
names = pi.groupby("institution_id")["institution_name"].first()
inst, s_i, top_i_strict = compare(eu_i, "institution_id", label=names)
venues_df = w[w["source_name"].notna()]
venues, s_v, top_v_strict = compare(venues_df, "source_name")
pd.DataFrame({"countries (EU)": s_c, "institutions (EU)": s_i, "venues": s_v}).T

,shared_top,jaccard,spearman_union_top
countries (EU),20.0,1.000,0.91
institutions (EU),13.0,0.481,0.65
venues,13.0,0.481,0.36


In [5]:
countries

,papers_ALL,rank_ALL,papers_STRICT,rank_STRICT
institution_country,,,,
DE,1764,1,1123,1
ES,1555,2,1061,2
IT,1140,3,647,3
FR,628,4,297,8
GR,623,5,309,7
FI,586,6,407,4
NL,562,7,390,5
PL,539,8,275,10
PT,482,9,280,9


In [6]:
inst

,name,papers_ALL,rank_ALL,papers_STRICT,rank_STRICT
institution_id,,,,,
https://openalex.org/I175532246,University of Eastern Finland,169,1,128,1
https://openalex.org/I62916508,Technical University of Munich,115,2,86,2
https://openalex.org/I8204097,Ludwig-Maximilians-Universität München,109,3,85,3
https://openalex.org/I98381234,University of Oulu,90,4,74,4
https://openalex.org/I100930933,University College Dublin,90,4,62,6
https://openalex.org/I1294671590,Centre National de la Recherche Scientifique,89,6,47,14
https://openalex.org/I181343428,University of Zagreb,85,7,58,7
https://openalex.org/I184999862,Universidad de Salamanca,81,8,68,5
https://openalex.org/I69050122,Near East University,78,9,44,22


In [7]:
print("Leave the venue top 20 in STRICT:")
print(sorted(set(venues.index) - set(top_v_strict)))
venues

Leave the venue top 20 in STRICT:
['Advances in computational intelligence and robotics book series', 'HAL', 'Journal of Inclusive Methodology and Technology in Learning and Teaching', 'SSRN Electronic Journal', 'Springer eBooks', 'Zenodo (CERN European Organization for Nuclear Research)', 'arXiv (Cornell University)']


,papers_ALL,rank_ALL,papers_STRICT,rank_STRICT
source_name,,,,
Lecture notes in computer science,210,1,210,1
arXiv (Cornell University),208,2,0,1647
Lecture notes in networks and systems,141,3,141,2
HAL,131,4,0,1647
Education Sciences,116,5,116,3
Communications in computer and information science,115,6,115,4
Applied Sciences,111,7,111,5
SSRN Electronic Journal,95,8,0,1647
IEEE Global Engineering Education Conference (EDUCON),92,9,92,6


## 3. Why institutions move: the biggest losers

In [8]:
A = eu_i.groupby("institution_id")["work_id"].nunique()
S = eu_i[eu_i["in_strict"]].groupby("institution_id")["work_id"].nunique().reindex(A.index, fill_value=0)
loss = pd.DataFrame({"name": names.reindex(A.index), "ALL": A, "STRICT": S, "lost": A - S,
                     "rank_ALL": A.rank(ascending=False, method="min").astype(int),
                     "rank_STRICT": S.rank(ascending=False, method="min").astype(int)})
loss = loss[loss["rank_ALL"] <= 30].sort_values("lost", ascending=False).head(10)
loss

,name,ALL,STRICT,lost,rank_ALL,rank_STRICT
institution_id,,,,,,
https://openalex.org/I194019607,University of Ioannina,65,14,51,20,195
https://openalex.org/I1294671590,Centre National de la Recherche Scientifique,89,47,42,6,14
https://openalex.org/I175532246,University of Eastern Finland,169,128,41,1,1
https://openalex.org/I200777214,National and Kapodistrian University of Athens,67,29,38,19,72
https://openalex.org/I88491126,Bucharest University of Economic Studies,73,36,37,12,38
https://openalex.org/I204337017,Aarhus University,73,37,36,12,35
https://openalex.org/I69050122,Near East University,78,44,34,9,22
https://openalex.org/I60053951,Universitat Politècnica de València,64,31,33,21,63
https://openalex.org/I61641377,Universitatea Națională de Știință și Tehnolog...,61,30,31,24,66


In [9]:
# Why the papers of the two largest losers drop out
wl = w.set_index("work_id")
def reasons(inst_id):
    ids = eu_i.loc[(eu_i["institution_id"] == inst_id) & ~eu_i["in_strict"], "work_id"]
    sub = wl.loc[ids]
    why = np.select([sub["is_repository"],
                     sub["in_scopus"] & ~sub["scopus_covered_in_year"],
                     sub["match_method"].eq("none") & sub["issn_l"].notna(),
                     sub["match_method"].eq("none") & sub["issn_l"].isna()],
                    ["repository", "Scopus, not covered that year",
                     "journal not in Scopus", "no ISSN, venue not matched"], "other")
    return pd.Series(why).value_counts()
pd.DataFrame({names[i]: reasons(i) for i in loss.index[:2]}).fillna(0).astype(int)

,University of Ioannina,Centre National de la Recherche Scientifique
"Scopus, not covered that year",1,3
journal not in Scopus,48,6
"no ISSN, venue not matched",1,17
repository,1,16


## 4. Do countries survive differently, and why?

In [10]:
cs = eu_c.groupby("institution_country").agg(papers=("work_id", "nunique"),
                                             strict=("in_strict", "sum"))
conf = eu_c.merge(w[["work_id", "type"]], on="work_id").assign(conf=lambda d: d["type"].eq("conference-paper"))
cs["conference_share"] = conf.groupby("institution_country")["conf"].mean()
cs["survival"] = cs["strict"] / cs["papers"]
cs = cs[cs["papers"] >= 50].sort_values("survival")
print(f"{len(cs)} countries with >= 50 papers; correlation of survival with conference share: "
      f"r = {cs['survival'].corr(cs['conference_share']):.2f}")
cs.round(2)

25 countries with >= 50 papers; correlation of survival with conference share: r = 0.09


,papers,strict,conference_share,survival
institution_country,,,,
BG,298,113,0.42,0.38
RO,473,189,0.25,0.40
FR,628,297,0.32,0.47
GR,623,309,0.23,0.50
HU,271,135,0.25,0.50
PL,539,275,0.19,0.51
LV,92,49,0.49,0.53
LT,103,56,0.22,0.54
IT,1140,647,0.26,0.57


## 5. Sensitivity: the country ranking under every STRICT variant

In [11]:
rows = {}
for v in VARIANTS:
    _, s, _ = compare(eu_c, "institution_country", strict_col=v)
    _, s2, _ = compare(eu_i, "institution_id", strict_col=v)
    rows[v] = {"papers": int(w[v].sum()), "countries_spearman": s["spearman_union_top"],
               "countries_shared_top20": s["shared_top"], "institutions_spearman": s2["spearman_union_top"],
               "institutions_shared_top20": s2["shared_top"]}
pd.DataFrame(rows).T

,papers,countries_spearman,countries_shared_top20,institutions_spearman,institutions_shared_top20
in_strict,5927.0,0.91,20.0,0.65,13.0
in_strict_issn_only,5262.0,0.93,20.0,0.35,11.0
in_strict_active,5824.0,0.90,20.0,0.62,13.0
in_strict_lag,5987.0,0.91,20.0,0.62,13.0
in_strict_q12,3886.0,0.92,19.0,0.12,10.0


## 6. What this shows

*Numbers in this section are filled from the outputs above.*

- **STRICT keeps 58% of papers.** Survival depends on the document type: reviews 86%, articles 68%, conference
  papers 52%, book chapters 23%, preprints 0%.
- **Countries are stable.** The EU top 20 is the same set in both layers, with Spearman 0.91, and stays at
  0.90–0.93 under every main STRICT variant. Country survival is not explained by conference share (r ≈ 0.1); it
  depends on which journals each country's researchers use.
- **Institutions move.** Only 13 of the top 20 are shared, with Spearman 0.65 (Person 2 reported 0.64; the small difference comes from how ties are ranked). The largest losers drop
  mainly because their journals are not in Scopus, as with Ioannina, or because of repositories and unchecked
  conference venues, as with CNRS.
- **Venues move most** (Spearman 0.36). Repositories and book platforms leave the top 20.

**What could overturn it.** The STRICT layer rests on the hand check of 50 venues. If more of the unchecked
venues are in Scopus, STRICT grows, and the institution ranking moves back towards ALL.